In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
MAZE = ["S.#.....", ".##.###.", "....#...", ".####.#.", "......#.", ".####.#G", "....#...", "###...#."]
G = np.array([list(r) for r in MAZE])
R, C = G.shape
START, GOAL = (0, 0), (5, 7)
D = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ARR = "↑↓←→"
alpha, gamma, eps, eps_min, decay, EP, MAXS = 0.1, 0.95, 1.0, 0.01, 0.995, 2000, 200
rng = np.random.default_rng(42)

In [ ]:
def step(s, a):
    r, c = s[0] + D[a][0], s[1] + D[a][1]
    if not (0 <= r < R and 0 <= c < C) or G[r, c] == "#":
        return s, -5, False
    g = (r, c) == GOAL
    return (r, c), 100 if g else -1, g

In [ ]:
Q = np.zeros((R, C, 4))
rew, stp = [], []

for ep in range(EP):
    s, tot = START, 0
    for t in range(MAXS):
        q = Q[s]
        a = int(rng.integers(4)) if rng.random() < eps else int(rng.choice(np.flatnonzero(q == q.max())))
        n, r, done = step(s, a)
        Q[s][a] += alpha * (r + (0 if done else gamma * Q[n].max()) - Q[s][a])
        s, tot = n, tot + r
        if done:
            break
    eps = max(eps_min, eps * decay)
    rew.append(tot)
    stp.append(t + 1)

In [ ]:
path, s = [START], START
while s != GOAL and len(path) < 64:
    s = step(s, int(np.argmax(Q[s])))[0]
    path.append(s)

print("Reached goal:", s == GOAL, "| Path length (steps):", len(path) - 1)
print("Path:", path)

In [ ]:
ma = lambda x, k=50: np.convolve(x, np.ones(k) / k, "valid")
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

for i, (d, t, l) in enumerate([(rew, "Reward per episode", "Episode reward"), (stp, "Steps to finish episode", "Steps")]):
    ax[i].plot(d, alpha=0.3, label=l)
    ax[i].plot(range(49, EP), ma(d), label="Moving avg (50)")
    ax[i].set(title=t, xlabel="Episode")
    ax[i].legend()

In [ ]:
a = ax[2]
a.imshow(G == "#", cmap="Greys", vmin=0, vmax=1.5)

for r in range(R):
    for c in range(C):
        if G[r, c] != "#":
            ch, col = (G[r, c], "green" if G[r, c] == "S" else "red") if G[r, c] in "SG" else (ARR[Q[r, c].argmax()], "steelblue")
            a.text(c, r, ch, ha="center", va="center", color=col, fontsize=14, fontweight="bold" if ch in "SG" else None)

a.plot(*zip(*[(c, r) for r, c in path]), "o-", color="orange", label="Greedy path")
a.set(title="Learned policy & optimal path", xticks=range(C), yticks=range(R))
a.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("qlearning_maze_results.png", dpi=150)
print("Saved plot to qlearning_maze_results.png")